# VoltSync AI — Real Data Run (Colab / Kaggle)**Hamza Naveed · 35017370 · MSc Data Science and AI · 55-710248**Supervisor: Dr. Efosa OsagieRuns the forecasting experiment on the **real** UK datasets, satisfyingObjective (ii): *obtain and prepare suitable datasets of UK householdelectricity demand and rooftop solar generation from publicly availablesources.*Use **Kaggle** for long runs — background execution survives a closed tab.Per-unit results are cached in `results/partial/`, so an interrupted runresumes where it stopped.

## 1 · Setup

In [ ]:
!pip -q install tensorflow xgboost polars huggingface_hub pyarrowimport tensorflow as tfprint("TF", tf.__version__, "| GPU:", bool(tf.config.list_physical_devices('GPU')))

In [ ]:
%cd /content# !git clone https://github.com/<you>/voltsync-forecast.git%cd voltsync-forecast!python tests/test_pipeline.py          # 21 correctness checks before we start

## 2 · Download uk_pv**Gated dataset.** Log in to Hugging Face, open the `uk_pv` page, click toaccept the conditions, then create a token.Do **not** call `load_dataset("openclimatefix/uk_pv")` — it raises`DatasetGenerationError`. Download the parquet directly.

In [ ]:
from huggingface_hub import loginlogin(token="hf_xxxxxxxxxxxxxxxx")from src.data import download_uk_pvpv_path = download_uk_pv(years=(2022, 2023))print(pv_path)!du -sh {pv_path}

## 3 · Download IDEAL demand dataPullinger et al. (2021), DOI 10.7488/ds/2836, handle 10283/3647.Resolve the bitstream list first — the web page uses bot detection, the RESTAPI does not.

In [ ]:
!curl -s "https://datashare.ed.ac.uk/rest/handle/10283/3647/bitstreams" | head -c 3000

In [ ]:
from src.data import download_ideal# Fill in the filenames returned above. Electricity files are gzipped CSVs.targets = {    # 65: "home65_electric_combined.csv.gz",    # 73: "home73_electric_combined.csv.gz",}ideal_files = {hid: download_ideal(handle="10283/3647", filename=fn, sequence=1)               for hid, fn in targets.items()}print(ideal_files)

## 4 · Switch to real data`load_data()` **refuses to fall back** to synthetic when `DATA_MODE="real"`, soa missing path raises instead of silently mislabelling your results.

In [ ]:
import src.config as CC.DATA_MODE = "real"print("DATA_MODE =", C.DATA_MODE)

In [ ]:
from src.data import load_datapv, demand, meta = load_data(n_pv=8, n_homes=8,                             pv_path=pv_path, ideal_files=ideal_files)print(pv.shape, demand.shape)meta[["ss_id","kWp","tilt","orientation","latitude_rounded","longitude_rounded"]]

In [ ]:
# Cleaning log -> Table 4.0 of the dissertationimport pandas as pdpd.DataFrame(pv.attrs["cleaning_log"], columns=["step", "rows"])

## 5 · Run the experimentWeather is fetched **per system** from its own coordinates (3.3.3), memoisedper ~25 km reanalysis grid cell, and sampled at the interval midpoint because`datetime_GMT` is period-ending.

In [ ]:
from run_experiment import maindem_res, sol_res, summary = main(n_homes=8, n_systems=8,                                 pv_path=pv_path, ideal_files=ideal_files)summary

## 6 · Results

In [ ]:
dem_res[["unit","mae","rmse","mape","baseline_mae","baseline_mape",         "mae_improvement_pct","beats_baseline"]].round(4)

In [ ]:
sol_res[["unit","kwp","mae","rmse","mape_daylight","baseline_mae",         "baseline_mape","mae_improvement_pct","beats_baseline"]].round(4)

In [ ]:
from IPython.display import Image, displayimport globfor f in sorted(glob.glob("figs/*.png")):    print(f); display(Image(f))

## 7 · Export

In [ ]:
!zip -qr voltsync_real_results.zip results/ figs/ models/from google.colab import files; files.download('voltsync_real_results.zip')